In [ ]:
import math
import pandas as pd


data = {
    'Study Hours': ['High','Low','Medium','Low','High','Medium','Low','High','Medium','Low','High','Medium','Low','High'],
    'Attendance': ['Good','Poor','Good','Poor','Good','Poor','Good','Good','Good','Poor','Poor','Good','Poor','Good'],
    'Sleep Quality': ['Enough','Less','Less','Enough','Enough','Enough', 'Less','Less','Enough','Less','Enough','Less', 'Enough','Enough'],
    'Group Study': ['Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'Yes'],
    'Exam Result': ['Pass', 'Fail', 'Pass', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass'],
}

df = pd.DataFrame(data)

df.show()


In [ ]:

# --------- ID3 Implementation ---------
def entropy(column):
    values, counts = pd.Series(column).value_counts(normalize=True).index, pd.Series(column).value_counts(normalize=True).values
    return -sum(p * math.log2(p) for p in counts)

def info_gain(df, attribute, target='Purchase'):
    total_entropy = entropy(df[target])
    values = df[attribute].unique()
    weighted_entropy = 0
    for val in values:
        subset = df[df[attribute] == val]
        weighted_entropy += len(subset)/len(df) * entropy(subset[target])
    return total_entropy - weighted_entropy

def predict(tree, sample):
    if not isinstance(tree, dict):  # nếu là lá -> trả kết quả
        return tree
    
    # lấy root attribute (ví dụ: Outlook, Temp...)
    root = next(iter(tree))
    value = sample[root]
    
    # nếu giá trị tồn tại trong nhánh
    if value in tree[root]:
        return predict(tree[root][value], sample)
    else:
        return None  # trường hợp không có trong training

def id3(df, target='Purchase', features=None):
    if features is None:
        features = df.columns.drop(target)

    # Nếu tất cả cùng 1 nhãn
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # Nếu hết thuộc tính
    if len(features) == 0:
        return df[target].mode()[0]

    # Chọn thuộc tính có information gain cao nhất
    gains = {feat: info_gain(df, feat, target) for feat in features}
    print(gains)
    best_feat = max(gains, key=gains.get)

    tree = {best_feat: {}}
    for val in df[best_feat].unique():
        subset = df[df[best_feat] == val]
        if subset.empty:
            tree[best_feat][val] = df[target].mode()[0]
        else:
            tree[best_feat][val] = id3(subset, target, [f for f in features if f != best_feat])
    return tree

def print_tree(tree, indent=""):
    if isinstance(tree, dict):
        for attr, branches in tree.items():
            print(f"{indent}{attr}")
            for val, subtree in branches.items():
                print(f"{indent}  └─ {val}:")
                print_tree(subtree, indent + "     ")
    else:
        print(f"{indent}  → {tree}")

# --------- Gọi hàm in cây ---------
tree = id3(df, target='Purchase')
print("Decision Tree (ID3):")
print_tree(tree)


{'Income': 0.7142857142857143, 'OnlineAd': 0.0, 'Discount': 0.5087256743873155, 'Brand': 0.6532319305519041}
{'OnlineAd': 0.31127812445913283, 'Discount': 1.0, 'Brand': 0.31127812445913283}
Decision Tree (ID3):
Income
  └─ high:
       → yes
  └─ low:
       → no
  └─ medium:
     Discount
       └─ yes:
            → yes
       └─ no:
            → no


In [ ]:
sample1 = {'Outlook': 'sunny', 'Temp': 'cool', 'Humidity': 'normal', 'Wind': 'weak'}
sample2 = {'Outlook': 'rain', 'Temp': 'mild', 'Humidity': 'high', 'Wind': 'strong'}

print("Sample1:", predict(tree, sample1))
print("Sample2:", predict(tree, sample2))

In [3]:
df

,Outlook,Temp,Humidity,Wind,Activity
0,rain,hot,high,strong,stay home
1,overcast,cool,high,strong,stay home
2,overcast,cool,normal,strong,walk
3,rain,cool,normal,strong,stay home
4,sunny,cool,normal,strong,tennis
5,sunny,cool,normal,weak,tennis
6,rain,hot,normal,strong,stay home
7,sunny,hot,normal,weak,walk
8,sunny,mild,normal,strong,tennis
9,sunny,mild,high,weak,tennis
